# HSBC External Search — Web Grounding Test Harness

## Manager-defined test scope

### Part A — URL / content extraction
- **A1:** Scrape the entire URL or not.
- **A2:** Extract images/tables inside the URL.
- **A3:** Extract/understand the structure of the content in the URL.
- **A4:** Scrape URLs/hyperlinks, including hyperlinks to PDFs.
- **A5:** Summarise the parent URL.
- **A6:** Summarise child URLs / associated PDFs.

### Part B — Regulatory intelligence
- **B1:** Given a URL, identify topics/themes and related URLs, with or without jurisdiction.
- **B2:** Given a URL, identify amendments, related news and the latest N amendments.

## Reporting approach

The notebook separates:
1. **Direct web diagnostics** — what the workstation can retrieve directly.
2. **External Search** — what the enterprise search service retrieves.
3. **Evidence** — citations, generated queries, tokens and timing.
4. **Executive outcome** — `PASS`, `PARTIAL`, `FAIL`, or `NOT TESTABLE`.

> This notebook does not bypass CAPTCHA, bot protection, authentication or other access controls. Such conditions are recorded as source-access limitations.

## Output files

- `Web_Grounding_Manager_Report.xlsx`
- `Web_Grounding_Manager_Summary.txt`

## 1. Environment and Configuration

Run this section first. Update `CERT_PATH` only if the corporate certificate is stored elsewhere.

In [ ]:
import requests
import uuid
import json
import re
import os
import time
import traceback
from getpass import getpass
from html.parser import HTMLParser
from urllib.parse import urljoin
import pandas as pd

BASE_URL = (
    "https://gaip-api-uat.hsbc-12152296-gaipukuat-dev.dev.gcp.cloud.uk.hsbc/"
    "aip-external-search-backend-uat-internal-proxy/v1"
)

SEARCH_ENGINE = "enterprise_web_search"
USE_CASE_ID = "UC0008068"
MODEL_NAME = "gemini-2.5-flash"

CHECK_MODEL_URL = f"{BASE_URL}/api/check_models/v1/{SEARCH_ENGINE}"
SEARCH_URL = f"{BASE_URL}/api/search/v1/{SEARCH_ENGINE}/{USE_CASE_ID}"

CERT_PATH = r"C:\Temp\certs\truststore-prod-2.0.7.pem"

REQUEST_TIMEOUT = 180
DIRECT_TIMEOUT = 30
MAX_CITATIONS = 10
MAX_CHILD_URLS = 10
DEFAULT_N_AMENDMENTS = 5

print("Environment and configuration loaded.")
print("Search URL:", SEARCH_URL)

## 2. Authentication

Enter the temporary AM token interactively. Do not hard-code it into the notebook.

In [ ]:
AM_TOKEN = getpass("Paste temporary AM token: ").strip()

if not AM_TOKEN:
    raise ValueError("AM token is empty.")

def create_headers():
    return {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {AM_TOKEN}",
        "x-correlation-id": str(uuid.uuid4()),
        "x-conversation-id": str(uuid.uuid4())
    }

print("Authentication headers prepared.")

## 3. Pre-flight model check

This validates the requested model before running a test.

In [ ]:
def check_model():
    try:
        r = requests.get(
            CHECK_MODEL_URL,
            headers=create_headers(),
            verify=CERT_PATH,
            timeout=60
        )
        data = r.json()
        models = data.get("available_model_list", [])
        return {
            "http_status": r.status_code,
            "available_models": models,
            "model_available": MODEL_NAME in models,
            "error": None
        }
    except Exception as e:
        return {
            "http_status": None,
            "available_models": [],
            "model_available": False,
            "error": str(e)
        }

model_check = check_model()
print(json.dumps(model_check, indent=2))

if not model_check["model_available"]:
    raise RuntimeError(
        f"{MODEL_NAME} is not available. Resolve this before testing."
    )

## 4. Direct URL diagnostics

This layer records source-level evidence: HTTP status, redirects, title, headings, links, PDFs, images, tables, text volume and common verification indicators.

A direct HTTP failure does **not** by itself mean External Search failed.

In [ ]:
class PageParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links, self.pdf_links, self.images = [], [], []
        self.headings, self.tables, self.text_chunks = [], [], []
        self.current_heading = None
        self.current_heading_text = []
        self.in_table = False
        self.current_table = []
        self.current_row = []
        self.in_row = False

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        tag = tag.lower()

        if tag == "a" and attrs.get("href"):
            href = attrs["href"]
            self.links.append(href)
            if ".pdf" in href.lower():
                self.pdf_links.append(href)

        elif tag == "img":
            src = attrs.get("src") or attrs.get("data-src")
            if src:
                self.images.append(src)

        elif tag in ["h1","h2","h3","h4","h5","h6"]:
            self.current_heading = tag
            self.current_heading_text = []

        elif tag == "table":
            self.in_table = True
            self.current_table = []

        elif tag == "tr" and self.in_table:
            self.in_row = True
            self.current_row = []

    def handle_endtag(self, tag):
        tag = tag.lower()

        if tag in ["h1","h2","h3","h4","h5","h6"]:
            text = " ".join(self.current_heading_text).strip()
            if text:
                self.headings.append({"level": self.current_heading, "text": text})
            self.current_heading = None
            self.current_heading_text = []

        elif tag == "tr":
            if self.in_table and self.current_row:
                self.current_table.append(self.current_row)
            self.current_row = []
            self.in_row = False

        elif tag == "table":
            if self.in_table and self.current_table:
                self.tables.append(self.current_table)
            self.current_table = []
            self.in_table = False

    def handle_data(self, data):
        clean = re.sub(r"\s+", " ", data).strip()
        if not clean:
            return
        self.text_chunks.append(clean)
        if self.current_heading:
            self.current_heading_text.append(clean)
        if self.in_row:
            self.current_row.append(clean)


def inspect_url(url):
    out = {
        "url": url,
        "http_status": None,
        "accessible": False,
        "final_url": None,
        "content_type": None,
        "content_bytes": None,
        "page_title": None,
        "headings": [],
        "heading_count": 0,
        "links": [],
        "link_count": 0,
        "pdf_links": [],
        "pdf_link_count": 0,
        "images": [],
        "image_count": 0,
        "tables": [],
        "table_count": 0,
        "text_length": 0,
        "verification_detected": [],
        "error": None
    }

    try:
        r = requests.get(
            url,
            timeout=DIRECT_TIMEOUT,
            allow_redirects=True,
            headers={"User-Agent": "Mozilla/5.0"},
            verify=CERT_PATH
        )

        out["http_status"] = r.status_code
        out["final_url"] = r.url
        out["content_type"] = r.headers.get("Content-Type")
        out["content_bytes"] = len(r.content)

        lower = r.text.lower()
        terms = [
            "captcha","recaptcha","hcaptcha","human verification",
            "verify you are human","checking your browser","cloudflare",
            "challenge-platform","access denied","bot verification",
            "security check","robot check"
        ]
        out["verification_detected"] = [x for x in terms if x in lower]

        out["accessible"] = (
            r.status_code == 200 and not out["verification_detected"]
        )

        if "application/pdf" in (out["content_type"] or "").lower():
            out["text_length"] = len(r.content)
            return out

        p = PageParser()
        p.feed(r.text)

        out["headings"] = p.headings
        out["heading_count"] = len(p.headings)
        out["links"] = list(dict.fromkeys(
            urljoin(r.url, x) for x in p.links
        ))
        out["link_count"] = len(out["links"])
        out["pdf_links"] = list(dict.fromkeys(
            urljoin(r.url, x) for x in p.pdf_links
        ))
        out["pdf_link_count"] = len(out["pdf_links"])
        out["images"] = list(dict.fromkeys(
            urljoin(r.url, x) for x in p.images
        ))
        out["image_count"] = len(out["images"])
        out["tables"] = p.tables
        out["table_count"] = len(p.tables)
        out["text_length"] = len(" ".join(p.text_chunks))

        m = re.search(r"<title[^>]*>(.*?)</title>", r.text, re.I | re.S)
        if m:
            out["page_title"] = re.sub(r"\s+", " ", m.group(1)).strip()

    except Exception as e:
        out["error"] = str(e)

    return out

## 5. External Search API wrapper

All manager-defined tests use this wrapper. It captures API status, `valid_search`, answer, generated queries, citations, grounding supports, tokens, timing and raw response.

In [ ]:
def external_search(question, whitelist=None, blacklist=None):
    start = time.perf_counter()

    result = {
        "http_status": None,
        "api_success": False,
        "valid_search": False,
        "answer": "",
        "search_queries": [],
        "citations": [],
        "citation_domains": [],
        "grounding_supports": [],
        "input_tokens": None,
        "output_tokens": None,
        "thoughts_tokens": None,
        "cached_tokens": None,
        "total_tokens": None,
        "processing_time": None,
        "latency_seconds": None,
        "error": None,
        "raw_response": None
    }

    payload = {
        "query": question,
        "model_name": MODEL_NAME,
        "search_engine_type": SEARCH_ENGINE,
        "domain_whitelist": whitelist,
        "domain_blacklist": blacklist,
        "max_citations": MAX_CITATIONS
    }

    try:
        r = requests.post(
            SEARCH_URL,
            headers=create_headers(),
            json=payload,
            verify=CERT_PATH,
            timeout=REQUEST_TIMEOUT
        )

        result["http_status"] = r.status_code
        result["latency_seconds"] = time.perf_counter() - start

        try:
            raw = r.json()
        except Exception:
            raw = {"raw_text": r.text}

        result["raw_response"] = raw
        result["api_success"] = r.status_code == 200

        data = raw.get("data", {})
        result["answer"] = data.get("answer") or ""

        metadata = data.get("search_metadata", {})
        result["valid_search"] = bool(metadata.get("valid_search", False))

        result["search_queries"] = data.get("search_queries", [])
        result["citations"] = data.get("citations", [])

        result["citation_domains"] = list(dict.fromkeys(
            c.get("domain")
            for c in result["citations"]
            if c.get("domain")
        ))

        result["grounding_supports"] = [
            support
            for c in result["citations"]
            for support in (c.get("grounding_supports") or [])
        ]

        tokens = data.get("token_usage_metadata", {})
        for k in [
            "input_tokens","output_tokens","thoughts_tokens",
            "cached_tokens","total_tokens"
        ]:
            result[k] = tokens.get(k)

        result["processing_time"] = data.get("processing_time")

    except Exception as e:
        result["error"] = str(e)
        result["latency_seconds"] = time.perf_counter() - start

    return result


def extract_urls_from_text(text):
    if not text:
        return []
    urls = re.findall(r'https?://[^\s<>"\')\]]+', text)
    return list(dict.fromkeys(x.rstrip(".,;:)") for x in urls))

# 6. Part A — Six manager-defined tests

### A1 — Scrape the entire URL or not

In [ ]:
def test_A1(url):
    inventory = inspect_url(url)

    prompt = "\n".join([
        "You are testing an enterprise web-grounding system.",
        f"TARGET URL: {url}",
        "Determine whether the webpage can be comprehensively retrieved.",
        "Identify page title, major sections, subsections, lists, tables, links, PDF links, images/figures and any evidence of partial retrieval.",
        "Return headings: PAGE TITLE; MAJOR SECTIONS; SUBSECTIONS; LISTS; TABLES; LINKS; PDF LINKS; IMAGES / FIGURES; CONTENT COMPLETENESS; LIMITATIONS.",
        "For CONTENT COMPLETENESS explicitly use COMPLETE / PARTIAL / UNKNOWN.",
        "Use retrieved web information only."
    ])

    search = external_search(prompt)
    text = search["answer"].upper()

    if "COMPLETE" in text and "PARTIAL" not in text:
        outcome = "PASS"
    elif "PARTIAL" in text:
        outcome = "PARTIAL"
    elif not search["valid_search"]:
        outcome = "FAIL"
    else:
        outcome = "UNKNOWN / REVIEW"

    return {
        "test_id": "A1",
        "test_name": "Scrape entire URL",
        "url": url,
        "outcome": outcome,
        "key_takeaway": "Tests whether External Search can retrieve the parent page comprehensively.",
        "evidence": {
            "direct_page_access": inventory["accessible"],
            "direct_http_status": inventory["http_status"],
            "page_title": inventory["page_title"],
            "local_heading_count": inventory["heading_count"],
            "local_link_count": inventory["link_count"],
            "local_pdf_count": inventory["pdf_link_count"],
            "local_image_count": inventory["image_count"],
            "local_table_count": inventory["table_count"],
            "valid_search": search["valid_search"],
            "citation_count": len(search["citations"])
        },
        "search": search,
        "inventory": inventory
    }

### A2 — Images / tables inside URL

In [ ]:
def test_A2(url):
    inventory = inspect_url(url)

    prompt = "\n".join([
        "You are testing image and table extraction.",
        f"WEBPAGE: {url}",
        "Investigate HTML tables, images, charts, figures and diagrams.",
        "For each element that you can retrieve, identify the information it contains.",
        "If actual visual/table content cannot be accessed, say so explicitly.",
        "Return TABLE EXTRACTION; IMAGE / FIGURE EXTRACTION; CHART EXTRACTION; DIAGRAM EXTRACTION; LIMITATIONS; OVERALL RESULT.",
        "For OVERALL RESULT use SUCCESS / PARTIAL / NOT AVAILABLE."
    ])

    search = external_search(prompt)
    text = search["answer"].upper()

    if inventory["image_count"] == 0 and inventory["table_count"] == 0:
        outcome = "NOT TESTABLE - NO DETECTED VISUALS"
    elif not search["valid_search"]:
        outcome = "FAIL"
    elif "SUCCESS" in text:
        outcome = "PASS"
    elif "PARTIAL" in text:
        outcome = "PARTIAL"
    else:
        outcome = "REVIEW"

    return {
        "test_id": "A2",
        "test_name": "Images / Tables extraction",
        "url": url,
        "outcome": outcome,
        "key_takeaway": "Tests whether information embedded in tables, images, charts and figures is retrievable.",
        "evidence": {
            "detected_images": inventory["image_count"],
            "detected_tables": inventory["table_count"],
            "valid_search": search["valid_search"],
            "citation_count": len(search["citations"])
        },
        "search": search,
        "inventory": inventory
    }

### A3 — Structure of content in URL

In [ ]:
def test_A3(url):
    inventory = inspect_url(url)

    prompt = "\n".join([
        "Analyse the structure of this webpage:",
        url,
        "Identify page title, H1/H2/H3, other headings, sections, subsections, lists, tables, nested content and logical relationships.",
        "Return TITLE; H1; H2; H3; OTHER HEADINGS; SECTION HIERARCHY; LISTS; TABLES; STRUCTURE RESULT; LIMITATIONS.",
        "For STRUCTURE RESULT use PRESERVED / PARTIAL / NOT PRESERVED."
    ])

    search = external_search(prompt)
    text = search["answer"].upper()

    if not search["valid_search"]:
        outcome = "FAIL"
    elif "PRESERVED" in text and "PARTIAL" not in text:
        outcome = "PASS"
    elif "PARTIAL" in text:
        outcome = "PARTIAL"
    else:
        outcome = "REVIEW"

    return {
        "test_id": "A3",
        "test_name": "Content structure extraction",
        "url": url,
        "outcome": outcome,
        "key_takeaway": "Tests whether page hierarchy and logical content structure can be understood.",
        "evidence": {
            "local_heading_count": inventory["heading_count"],
            "local_table_count": inventory["table_count"],
            "valid_search": search["valid_search"]
        },
        "search": search,
        "inventory": inventory
    }

### A4 — URL / hyperlinks / hyperlinks to PDFs

In [ ]:
def test_A4(url):
    inventory = inspect_url(url)

    prompt = "\n".join([
        "Test hyperlink traversal from this parent webpage:",
        url,
        "Check normal child webpages, PDF hyperlinks and linked documents.",
        "For relevant child links state PARENT LINK, CHILD URL, CONTENT TYPE, CONTENT RETRIEVED, KEY INFORMATION and LIMITATION.",
        "Also state whether PDF content could be retrieved.",
        "Overall result must be SUCCESS / PARTIAL / NOT AVAILABLE."
    ])

    search = external_search(prompt)
    text = search["answer"].upper()

    if not inventory["links"] and not inventory["pdf_links"]:
        outcome = "NOT TESTABLE - NO LINKS DETECTED"
    elif not search["valid_search"]:
        outcome = "FAIL"
    elif "SUCCESS" in text:
        outcome = "PASS"
    elif "PARTIAL" in text:
        outcome = "PARTIAL"
    else:
        outcome = "REVIEW"

    return {
        "test_id": "A4",
        "test_name": "Hyperlinks / PDF hyperlinks",
        "url": url,
        "outcome": outcome,
        "key_takeaway": "Tests whether External Search can move from a parent URL to linked webpages and PDFs.",
        "evidence": {
            "parent_links_detected": inventory["link_count"],
            "pdf_links_detected": inventory["pdf_link_count"],
            "sample_child_urls": inventory["links"][:10],
            "sample_pdf_urls": inventory["pdf_links"][:10],
            "valid_search": search["valid_search"],
            "citation_count": len(search["citations"])
        },
        "search": search,
        "inventory": inventory
    }

### A5 — Summarisation on parent URL

In [ ]:
def test_A5(url):
    prompt = "\n".join([
        "Summarise the content of this webpage:",
        url,
        "Base the summary on the supplied webpage. Do not use generic model knowledge.",
        "Identify page purpose, key topics, key facts, important dates and regulatory/business implications where applicable.",
        "Return PAGE PURPOSE; KEY TOPICS; KEY FACTS; IMPORTANT DATES; KEY IMPLICATIONS; SUMMARY; LIMITATIONS.",
        "Cite the retrieved source."
    ])

    search = external_search(prompt)

    outcome = (
        "PASS" if search["valid_search"] and search["answer"].strip()
        else "FAIL"
    )

    return {
        "test_id": "A5",
        "test_name": "Parent URL summarisation",
        "url": url,
        "outcome": outcome,
        "key_takeaway": "Tests whether the parent webpage can be summarised using grounded web content.",
        "evidence": {
            "valid_search": search["valid_search"],
            "citation_count": len(search["citations"])
        },
        "search": search,
        "inventory": inspect_url(url)
    }

### A6 — Summarisation on child URLs / associated PDFs

In [ ]:
def test_A6(url):
    inventory = inspect_url(url)

    associated = list(dict.fromkeys(
        inventory["links"][:MAX_CHILD_URLS] +
        inventory["pdf_links"][:MAX_CHILD_URLS]
    ))

    if not associated:
        discovery_prompt = "\n".join([
            "For this webpage identify important child URLs and associated PDF documents:",
            url,
            "Return the URLs and a short description of each."
        ])
        discovery = external_search(discovery_prompt)
        associated = extract_urls_from_text(discovery["answer"])[:MAX_CHILD_URLS]

    if not associated:
        return {
            "test_id": "A6",
            "test_name": "Child URLs / associated PDF summarisation",
            "url": url,
            "outcome": "NOT TESTABLE - NO CHILD URL FOUND",
            "key_takeaway": "No child webpage or associated PDF could be identified.",
            "evidence": {"child_urls": [], "pdf_urls": []},
            "search": {},
            "inventory": inventory
        }

    prompt = "\n".join([
        f"Parent webpage: {url}",
        "Associated child URLs / documents:",
        *associated,
        "Summarise important content from the child webpages and associated PDFs.",
        "For each source provide SOURCE URL, SOURCE TYPE, TITLE / DOCUMENT NAME, KEY TOPICS, KEY FACTS, IMPORTANT DATES, SUMMARY and LIMITATIONS.",
        "Do not invent information. If a child source cannot be retrieved, say NOT RETRIEVABLE."
    ])

    search = external_search(prompt)

    outcome = "PASS" if search["valid_search"] and search["answer"].strip() else "FAIL"

    return {
        "test_id": "A6",
        "test_name": "Child URLs / associated PDF summarisation",
        "url": url,
        "outcome": outcome,
        "key_takeaway": "Tests whether linked child webpages and associated PDFs can be summarised.",
        "evidence": {
            "child_urls_tested": associated,
            "valid_search": search["valid_search"],
            "citation_count": len(search["citations"])
        },
        "search": search,
        "inventory": inventory
    }

# 7. Part B — Regulatory intelligence

### B1 — Topics / themes / related URLs, with or without jurisdiction

In [ ]:
def test_B1(url, jurisdiction=None):
    jurisdiction_text = jurisdiction or "NOT PROVIDED"

    prompt = "\n".join([
        "Perform a regulatory web-grounding discovery test.",
        f"SOURCE REGULATION / WEBPAGE: {url}",
        f"JURISDICTION: {jurisdiction_text}",
        "Identify major regulatory topics, major themes, the regulation/instrument, related concepts and relevant related URLs.",
        "If jurisdiction is provided, keep discovery scoped to it.",
        "If jurisdiction is not provided, state whether jurisdiction can be inferred.",
        "Return SOURCE; JURISDICTION; REGULATION / INSTRUMENT; TOPICS; THEMES; RELATED REGULATORY URLs; JURISDICTION CONFIDENCE; LIMITATIONS.",
        "Do not invent URLs. Use citations."
    ])

    search = external_search(prompt)
    answer = search["answer"]
    related_urls = extract_urls_from_text(answer)

    if not search["valid_search"]:
        outcome = "FAIL"
    elif related_urls:
        outcome = "PASS"
    elif answer.strip():
        outcome = "PARTIAL"
    else:
        outcome = "FAIL"

    return {
        "test_id": "B1",
        "test_name": "Topics / themes / related URLs",
        "url": url,
        "outcome": outcome,
        "key_takeaway": "Tests whether a regulatory URL can seed topic/theme and related-URL discovery, with or without jurisdiction.",
        "evidence": {
            "jurisdiction": jurisdiction_text,
            "related_urls_found": related_urls,
            "citation_count": len(search["citations"]),
            "valid_search": search["valid_search"]
        },
        "search": search,
        "inventory": inspect_url(url)
    }

### B2 — Amendments / related news / latest N amendments

In [ ]:
def test_B2(url, n_amendments=DEFAULT_N_AMENDMENTS):
    prompt = "\n".join([
        "Perform a regulatory amendment intelligence test.",
        f"SOURCE REGULATION URL: {url}",
        "Identify the regulation/instrument, amendments, amendment dates, what changed, related regulatory/industry news and the latest requested amendments.",
        f"Return the latest {n_amendments} amendments.",
        "For each amendment provide Date, Title/identifier, What changed and Source.",
        "Then return RELATED NEWS, AMENDMENT COUNT RETURNED and LIMITATIONS.",
        "Do not invent amendments or news. If fewer than requested can be reliably identified, state the actual number."
    ])

    search = external_search(prompt)
    answer = search["answer"]

    matches = re.findall(r"AMENDMENT\s+\d+", answer, flags=re.I)
    amendment_count = len(set(x.upper() for x in matches))

    if not search["valid_search"]:
        outcome = "FAIL"
    elif amendment_count >= n_amendments:
        outcome = "PASS"
    elif amendment_count > 0 or answer.strip():
        outcome = "PARTIAL"
    else:
        outcome = "FAIL"

    return {
        "test_id": "B2",
        "test_name": "Amendments + related news + latest N",
        "url": url,
        "outcome": outcome,
        "key_takeaway": "Tests whether External Search can identify amendments, related news and the latest requested number of amendments.",
        "evidence": {
            "requested_amendments": n_amendments,
            "amendment_entries_detected": amendment_count,
            "citation_count": len(search["citations"]),
            "valid_search": search["valid_search"]
        },
        "search": search,
        "inventory": inspect_url(url)
    }

# 8. Test Runner

Choose exactly one of A1–A6, B1 or B2.

For **B1**, enter jurisdiction or leave it blank.
For **B2**, enter the number of latest amendments.

In [ ]:
def run_test(test_id, url, jurisdiction=None, n_amendments=DEFAULT_N_AMENDMENTS):
    if test_id == "A1": return test_A1(url)
    if test_id == "A2": return test_A2(url)
    if test_id == "A3": return test_A3(url)
    if test_id == "A4": return test_A4(url)
    if test_id == "A5": return test_A5(url)
    if test_id == "A6": return test_A6(url)
    if test_id == "B1": return test_B1(url, jurisdiction)
    if test_id == "B2": return test_B2(url, n_amendments)
    raise ValueError("Invalid test ID.")

print("""
A1 - Scrape entire URL
A2 - Images / Tables
A3 - Content structure
A4 - Hyperlinks / PDF hyperlinks
A5 - Parent URL summarisation
A6 - Child URL / associated PDF summarisation
B1 - Topics / themes / related URLs
B2 - Amendments / related news / latest N
""")

TEST_ID = input("Test ID: ").strip().upper()
TARGET_URL = input("Target URL: ").strip()

if TEST_ID not in ["A1","A2","A3","A4","A5","A6","B1","B2"]:
    raise ValueError("Invalid test ID.")

if not TARGET_URL.startswith(("http://", "https://")):
    raise ValueError("URL must start with http:// or https://")

JURISDICTION = None
N_AMENDMENTS = DEFAULT_N_AMENDMENTS

if TEST_ID == "B1":
    value = input("Jurisdiction (press Enter if not specified): ").strip()
    JURISDICTION = value or None

if TEST_ID == "B2":
    value = input(
        f"Number of latest amendments [default {DEFAULT_N_AMENDMENTS}]: "
    ).strip()
    if value:
        N_AMENDMENTS = int(value)

print("Ready:", TEST_ID, TARGET_URL)

# 9. Execute the selected test

This is the main execution cell.

In [ ]:
start_time = time.perf_counter()

try:
    result = run_test(TEST_ID, TARGET_URL, JURISDICTION, N_AMENDMENTS)
except Exception:
    print(traceback.format_exc())
    raise

total_runtime = time.perf_counter() - start_time
search = result.get("search", {})
inventory = result.get("inventory", {})

print("=" * 80)
print("TEST COMPLETE")
print("=" * 80)
print("TEST:", result["test_id"], "-", result["test_name"])
print("OUTCOME:", result["outcome"])
print()
print("KEY TAKEAWAY:")
print(result["key_takeaway"])
print()
print("VALID SEARCH:", search.get("valid_search"))
print("CITATIONS:", len(search.get("citations", [])))
print("RUNTIME:", round(total_runtime, 2), "seconds")
print()
print("ANSWER:")
print(search.get("answer", ""))

# 10. Reporting Layer

### Executive layer
The VP-facing result is intentionally limited to:
- Test
- Outcome
- Key takeaway

### Engineering evidence
The workbook also preserves:
- HTTP diagnostics
- verification indicators
- detected links/PDFs/images/tables
- generated search queries
- citations
- token usage
- processing time
- raw API response

This keeps management reporting concise without losing technical traceability.

In [ ]:
def build_report_tables(result, runtime_seconds):
    search = result.get("search", {})
    inventory = result.get("inventory", {})

    executive = pd.DataFrame([{
        "Test ID": result["test_id"],
        "Test": result["test_name"],
        "URL": result["url"],
        "Outcome": result["outcome"],
        "Key Takeaway": result["key_takeaway"]
    }])

    inventory_df = pd.DataFrame([{
        "URL": result["url"],
        "HTTP Status": inventory.get("http_status"),
        "Accessible": inventory.get("accessible"),
        "Final URL": inventory.get("final_url"),
        "Page Title": inventory.get("page_title"),
        "Content Type": inventory.get("content_type"),
        "Content Bytes": inventory.get("content_bytes"),
        "Heading Count": inventory.get("heading_count"),
        "Link Count": inventory.get("link_count"),
        "PDF Link Count": inventory.get("pdf_link_count"),
        "Image Count": inventory.get("image_count"),
        "Table Count": inventory.get("table_count"),
        "Text Length": inventory.get("text_length"),
        "Verification": ", ".join(inventory.get("verification_detected", [])),
        "Error": inventory.get("error")
    }])

    children = pd.DataFrame([
        {
            "Parent URL": result["url"],
            "Child URL": x,
            "Likely PDF": ".pdf" in x.lower()
        }
        for x in inventory.get("links", [])[:MAX_CHILD_URLS]
    ])

    citations = pd.DataFrame([
        {
            "Test ID": result["test_id"],
            "Citation #": i,
            "Domain": c.get("domain"),
            "URL": c.get("url"),
            "Grounding Supports": json.dumps(
                c.get("grounding_supports", []),
                default=str
            )
        }
        for i, c in enumerate(search.get("citations", []), 1)
    ])

    queries = pd.DataFrame([
        {
            "Test ID": result["test_id"],
            "Query #": i,
            "Generated Search Query": q
        }
        for i, q in enumerate(search.get("search_queries", []), 1)
    ])

    technical = pd.DataFrame([{
        "Test ID": result["test_id"],
        "Direct HTTP Status": inventory.get("http_status"),
        "Direct Access": inventory.get("accessible"),
        "Human Verification": ", ".join(inventory.get("verification_detected", [])),
        "External Search HTTP": search.get("http_status"),
        "API Success": search.get("api_success"),
        "Valid Search": search.get("valid_search"),
        "Citation Count": len(search.get("citations", [])),
        "Citation Domains": ", ".join(search.get("citation_domains", [])),
        "Input Tokens": search.get("input_tokens"),
        "Output Tokens": search.get("output_tokens"),
        "Total Tokens": search.get("total_tokens"),
        "Processing Time": search.get("processing_time"),
        "Client Runtime Seconds": round(runtime_seconds, 2)
    }])

    answer = pd.DataFrame([{
        "Test ID": result["test_id"],
        "Test": result["test_name"],
        "Answer": search.get("answer", "")
    }])

    raw = pd.DataFrame([{
        "Test ID": result["test_id"],
        "Raw API Response": json.dumps(
            search.get("raw_response"),
            indent=2,
            default=str
        )
    }])

    return {
        "EXECUTIVE_RESULT": executive,
        "TECHNICAL_EVIDENCE": technical,
        "PAGE_INVENTORY": inventory_df,
        "CHILD_LINKS": children,
        "CITATIONS": citations,
        "SEARCH_QUERIES": queries,
        "ANSWER": answer,
        "RAW_API_RESPONSE": raw
    }

report_tables = build_report_tables(result, total_runtime)
print("Report tables prepared.")

# 11. VP-friendly executive summary

The summary follows a simple pattern:

> **Outcome → key takeaway → implication**

Use the Excel `EXECUTIVE_RESULT` sheet for the management view.

In [ ]:
executive_text = "\n".join([
    "HSBC WEB GROUNDING TEST — EXECUTIVE RESULT",
    "",
    f"TEST: {result['test_id']} - {result['test_name']}",
    f"TARGET: {TARGET_URL}",
    f"OUTCOME: {result['outcome']}",
    "",
    "KEY TAKEAWAY:",
    result["key_takeaway"],
    "",
    "EXECUTIVE ANSWER:",
    search.get("answer", ""),
    "",
    "TECHNICAL EVIDENCE:",
    f"Direct HTTP Status: {inventory.get('http_status')}",
    f"Direct Access: {inventory.get('accessible')}",
    f"Human Verification: {inventory.get('verification_detected')}",
    f"External Search HTTP: {search.get('http_status')}",
    f"Valid Search: {search.get('valid_search')}",
    f"Citations: {len(search.get('citations', []))}",
    f"Runtime: {round(total_runtime, 2)} seconds"
])

print(executive_text)

# 12. Export Excel + text report

## Excel sheets

1. `EXECUTIVE_RESULT` — concise management view.
2. `TECHNICAL_EVIDENCE` — API and timing evidence.
3. `PAGE_INVENTORY` — direct source diagnostics.
4. `CHILD_LINKS` — detected child URLs.
5. `CITATIONS` — grounding evidence.
6. `SEARCH_QUERIES` — generated search queries.
7. `ANSWER` — full response.
8. `RAW_API_RESPONSE` — engineering trace.

In [ ]:
REPORT_FILE = "Web_Grounding_Manager_Report.xlsx"
SUMMARY_FILE = "Web_Grounding_Manager_Summary.txt"

with pd.ExcelWriter(REPORT_FILE, engine="openpyxl") as writer:
    for sheet_name, df in report_tables.items():
        if not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    f.write(executive_text)

print("Created:")
print(os.path.abspath(REPORT_FILE))
print(os.path.abspath(SUMMARY_FILE))

# 13. QA checklist before sending to management

### Execution
- [ ] Correct manager-defined test selected.
- [ ] Correct URL used.
- [ ] Correct jurisdiction supplied for B1 where applicable.
- [ ] Correct N supplied for B2.
- [ ] Model availability check passed.

### Evidence
- [ ] External Search HTTP status checked.
- [ ] `valid_search` checked.
- [ ] Citations reviewed.
- [ ] Search queries retained.
- [ ] Direct access limitation captured.
- [ ] CAPTCHA/human-verification limitation captured if present.

### Management communication
- [ ] Outcome is clear.
- [ ] Key takeaway is understandable without technical knowledge.
- [ ] No claim of "entire page scraped" based solely on a successful answer.
- [ ] No claim that direct HTTP failure means External Search failed.
- [ ] Material limitations are explicit.

## Interpretation

- **PASS** — capability demonstrated for the requested test.
- **PARTIAL** — capability demonstrated with a material limitation.
- **FAIL** — capability did not work for the test.
- **NOT TESTABLE** — required source artefact was not present.
- **UNKNOWN / REVIEW** — evidence was insufficient for a binary conclusion.

> A PASS means the specific test succeeded; it is not a production-readiness sign-off.